# 4. Teoria della Percolazione e Stress-Test di Resilienza
**Progetto:** La Topologia della Resilienza Urbana  
**Autore:** Urban Network Resilience Lab  
**Descrizione:** Questo notebook simula il collasso progressivo della rete di trasporto di Bologna sotto l'effetto di guasti casuali, incidenti stradali e attacchi mirati. Vengono analizzati i risultati delle simulazioni basate sulla teoria della percolazione e commentate le curve di resilienza.

## 4.1 I Tre Scenari di Attacco Ingegneristici

Per comprendere la robustezza strutturale del sistema, vengono simulate tre diverse strategie di collasso (rimozione progressiva dei nodi):

1. **Guasto Casuale (Random Failure):** Simula guasti tecnici sparsi e non coordinati. I nodi vengono rimossi tramite campionamento casuale uniforme.
2. **Attacco Mirato (Targeted Attack):** Simula il blocco coordinato dei punti neurali della mobilità (es. scioperi localizzati, grandi cantieri). I nodi vengono rimossi in ordine decrescente di Betweenness Centrality temporale.
3. **Guasto Basato sul Rischio Reale (Risk-Based Failure):** Supera la logica puramente teorica e mappa il collasso sulla base della sinistrosità stradale storica rilevata nel preprocessing. La probabilità di rimozione di ciascun nodo è pesata sul numero di incidenti nel raggio di 300 metri:
   $$P(\text{rimozione}_v) = \frac{\text{Incidenti}_v + 1.0}{\sum_{u \in V} (\text{Incidenti}_u + 1.0)}$$
   Il fattore $+1.0$ (*base risk*) garantisce che ogni fermata mantenga una probabilità infinitesima non nulla di subire guasti indipendenti dal traffico.

## 4.2 Il Calcolo della Vera Efficienza Globale Residua

Per quantificare il degrado del livello di servizio percepito dai cittadini, non ci si limita a monitorare la connettività geometrica della componente gigante (*LCC*). Viene invece ricalcolata a ogni iterazione la **Vera Efficienza Globale Pesata sul Tempo ($E(G)$)**:
$$E(G) = \frac{1}{N(N-1)} \sum_{u \neq v} \frac{1}{d_{\text{weighted}}(u,v)}$$
I cammini minimi temporali $d_{\text{weighted}}(u,v)$ vengono ricalcolati in secondi tramite Dijkstra su ogni grafo mutilato, fornendo una stima reale dei ritardi medi indotti dai detours stradali.

In [ ]:
import sys
import os
from pathlib import Path

# Aggiungiamo la root del progetto per importare src
sys.path.append(str(Path("..").resolve()))

from src.plot_resilience import plot_resilience_curves

print("--- Step 1: Esecuzione del Motore di Plotting di Resilienza ---")
# Esegue il caricamento del CSV generato dalla simulazione e aggiorna il grafico
plot_resilience_curves()

In [ ]:
from IPython.display import Image, display

print("\n--- Step 2: Visualizzazione delle Curve di Percolazione ---")
plot_path = "../data_output/bologna/bologna_resilience_curves.png"
if os.path.exists(plot_path):
    display(Image(filename=plot_path))
else:
    print("Avviso: Lancia prima la simulazione in run_analysis_bologna.py per esportare i risultati!")

## 4.3 Analisi e Interpretazione dei Risultati (La Verità delle Curve)

L'analisi del grafico e del file dei risultati rivela due dinamiche critiche:

* **Soglia di Percolazione Critica ($p_c$):** Sotto attacco mirato (linea rossa), la dimensione della componente gigante (*LCC*) subisce un crollo catastrofico e improvviso intorno al **25% di rimozione dei nodi** (l'integrità scende da `0.47` a `0.17`). Questo rappresenta il punto di rottura teorico della città di Bologna: eliminando un quarto degli hub principali, la rete si frammenta in micro-isole isolate non comunicanti.
* **Scissione tra Topologia e Dinamica (LCC vs Efficienza):** Al 5% di attacco mirato, la connettività geometrica (*LCC*) rimane quasi intatta (`0.91`), ma l'Efficienza Globale dei tempi di viaggio precipita immediatamente a `0.79`. Questo dimostra il paradosso dei sistemi di trasporto storici: la città non si spezza geometricamente (ci sono sempre vicoli secondari e percorsi alternativi per raggiungere la destinazione), ma l'efficienza temporale collassa perché i cittadini sono costretti a percorrere lunghe deviazioni congestionate a causa dell'inagibilità delle arterie primarie.